Please Note: Code outputs are displayed towards the end. We have built a pipeline and it is executed inside the last cell.

In [ ]:
#Import necessary libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import squarify
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from math import pi
import json
from datetime import datetime
import warnings

# filter noisy matplotlib warnings
warnings.filterwarnings("ignore")
plt.rcParams.update({'figure.max_open_warning': 0})


In [2]:
# DB connection configuration
DB_PARAMS = {
    "host": "ttc-data-traffic-data.b.aivencloud.com",
    "database": "defaultdb",
    "user": "group4ttc",
    "password": "Gr0up$004",
    "port": 25159
}

#Helper: Create SQLAlchemy engine
#----------------------------------------------------------
def connect_engine():
    """Returns a SQLAlchemy engine based on DB_PARAMS."""
    return create_engine(
        f"postgresql+psycopg2://{DB_PARAMS['user']}:{DB_PARAMS['password']}@"
        f"{DB_PARAMS['host']}:{DB_PARAMS['port']}/{DB_PARAMS['database']}?sslmode=require"
    )

engine = connect_engine()
print("Database engine created successfully.")

Database engine created successfully.


In [3]:
# KPI list used throughout the pipeline
KPI_LIST = [
    "riders_per_route",
    "riders_per_hour",
    "riders_per_trip",
    "operational_cost_per_rider",
    "operational_cost_per_day",
    "trip_duration",
    "route_duration",
    "load_factor_per_trip",
    "trips_per_day",
    "hours_per_day",
    "off_peak_trip_ratio",
    "service_span"
]

# Data Loading

def load_kpis():
    """Load KPI data from the fact table. Aggregate by mean if multiple rows per route."""
    
    sql = """
    SELECT
        route_id,
        riders_per_day    AS riders_per_route,
        riders_per_hour,
        riders_per_trip,
        operational_cost_per_rider,
        operational_cost_per_day,
        trip_duration,
        route_duration,
        load_factor_per_trip,
        trips_per_day,
        hours_per_day,
        off_peak_trip_ratio,
        service_span
    FROM gold."gold.fact_transit";
    """

    df = pd.read_sql_query(sql, engine)

    # Aggregate if multiple entries per route exist
    if df['route_id'].duplicated().any():
        df = df.groupby('route_id', as_index=False).mean()

    # convert route_id to string
    df['route_id'] = df['route_id'].astype(str)

    return df


In [4]:
# Outlier Detection, Handling & Sensitivity Analysis

#IQR-based outlier detection helper
def detect_outliers_iqr(data, column):
    """Detect outliers for a numeric column using the IQR method."""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_mask = (data[column] < lower_bound) | (data[column] > upper_bound)
    outliers_count = int(outliers_mask.sum())
    return outliers_count, lower_bound, upper_bound, outliers_mask

#full outlier detection across numeric columns
def perform_outlier_detection(df, skip_cols=None):
    if skip_cols is None:
        skip_cols = ['route_id']

    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    outlier_summary = []

    #loop through numeric columns and find IQR outliers
    for col in numeric_cols:
        try:
            outliers_count, lower_bound, upper_bound, out_mask = detect_outliers_iqr(df, col)
            outlier_pct = (outliers_count / len(df)) * 100
            status = "🟢 Clean" if outliers_count == 0 else "🔴 Found"
            action = "Flag for Winsorize" if outliers_count > 0 else "None"
            outlier_summary.append({
                'Column': col,
                'Total_Rows': len(df),
                'Outliers': outliers_count,
                'Outlier_%': round(outlier_pct, 3),
                'Lower_Bound': lower_bound,
                'Upper_Bound': upper_bound,
                'Status': status,
                'Action': action
            })
        except Exception as e:
            #if a column cannot be analyzed, record it as skipped
            outlier_summary.append({
                'Column': col,
                'Total_Rows': len(df),
                'Outliers': np.nan,
                'Outlier_%': np.nan,
                'Lower_Bound': np.nan,
                'Upper_Bound': np.nan,
                'Status': f"⚠ Skipped ({e})",
                'Action': "Skipped"
            })

    #build DataFrame and print a concise console summary
    outlier_df = pd.DataFrame(outlier_summary)
    print("\nOutlier Detection Summary (IQR)")
    print(outlier_df[['Column','Outliers','Outlier_%','Status','Action']].to_string(index=False))
    return outlier_df


In [5]:
#Winsorization
def handle_outliers_and_winsorize(df, lower_pct=0.01, upper_pct=0.99, skip_cols=None):
    """Apply winsorization to numeric columns and record caps per column."""
    if skip_cols is None:
        skip_cols = ['route_id']

    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    process_cols = [c for c in numeric_cols if c not in skip_cols]
    df_cleaned = df.copy()
    winsorization_summary = []

    #cap values at specified percentiles and track counts
    for col in numeric_cols:
        try:
            if col in process_cols:
                lower = df[col].quantile(lower_pct)
                upper = df[col].quantile(upper_pct)
                original = df[col].copy()
                df_cleaned[col] = df[col].clip(lower=lower, upper=upper)
                capped_count = int((original != df_cleaned[col]).sum())
                action = "Capped" if capped_count > 0 else "No Action"
                winsorization_summary.append({
                    'Column': col,
                    'Lower_Cap': lower,
                    'Upper_Cap': upper,
                    'Capped_Values': capped_count,
                    'Action': action
                })
            else:
                winsorization_summary.append({
                    'Column': col,
                    'Lower_Cap': np.nan,
                    'Upper_Cap': np.nan,
                    'Capped_Values': 0,
                    'Action': "Skipped (non-numeric or ID)"
                })
        except Exception as e:
            winsorization_summary.append({
                'Column': col,
                'Lower_Cap': np.nan,
                'Upper_Cap': np.nan,
                'Capped_Values': np.nan,
                'Action': f"Error: {e}"
            })

    #format summary DataFrame and return
    winsor_df = pd.DataFrame(winsorization_summary)
    return df_cleaned, winsor_df



In [6]:
# Sensitivity analysis
def perform_sensitivity_analysis(df_before, df_after, skip_cols=None):
    """Compare statistics before and after winsorization; produce status flags."""
    if skip_cols is None:
        skip_cols = ['route_id']

    numeric_cols = df_before.select_dtypes(include=['number']).columns.tolist()
    sensitivity_data = []

    #compute before & after stats and percent changes
    for col in numeric_cols:
        before = df_before[col].fillna(0).astype(float)
        after = df_after[col].fillna(0).astype(float)
        before_mean = before.mean()
        after_mean = after.mean()
        before_median = before.median()
        after_median = after.median()
        before_std = before.std()
        after_std = after.std()

        mean_change = ((after_mean - before_mean) / before_mean * 100) if before_mean != 0 else 0.0
        median_change = ((after_median - before_median) / before_median * 100) if before_median != 0 else 0.0
        std_change = ((after_std - before_std) / before_std * 100) if before_std != 0 else 0.0

        #decide status based on magnitude of mean change (same thresholds as before)
        if abs(mean_change) < 5:
            status = "🟢 Good"
        elif abs(mean_change) < 10:
            status = "🟡 Review"
        else:
            status = "🔴 High"

        sensitivity_data.append({
            'Column': col,
            'Before_Mean': before_mean,
            'After_Mean': after_mean,
            'Mean_Change_%': mean_change,
            'Before_Median': before_median,
            'After_Median': after_median,
            'Median_Change_%': median_change,
            'Before_Std': before_std,
            'After_Std': after_std,
            'Std_Change_%': std_change,
            'Range_Before': f"[{before.min():.3f}, {before.max():.3f}]",
            'Range_After': f"[{after.min():.3f}, {after.max():.3f}]",
            'Status': status
        })

    #compile into DataFrame and print concise summary
    sensitivity_df = pd.DataFrame(sensitivity_data)
    capped_total = int(((df_before != df_after).sum().sum()))
    print("\nSensitivity check done. Values capped:", capped_total)
    print(sensitivity_df[['Column','Mean_Change_%','Median_Change_%','Std_Change_%','Status']].to_string(index=False))
    return sensitivity_df  


In [7]:
#Outlier Detection & Handling Summary:
#The dataset undergoes outlier detection using the IQR method, flagging extreme KPI values that fall outside 1.5×IQR from the quartiles.
#These outliers are then handled via winsorization(except indentifiers), capping values at the 1st and 99th percentiles to reduce their impact without removing rows.
#A sensitivity analysis compares statistics before and after winsorization, ensuring the adjustments do not distort the data.


In [8]:
# Route Performance Analysis Visualizations

In [9]:
def run_route_performance_analysis(df_kpi, output_folder="results/route_performance_visuals"):
    """Generate comprehensive route performance visualizations."""
    os.makedirs(output_folder, exist_ok=True)
    
    df_sample = df_kpi.sort_values('trips_per_day', ascending=False).head(100)
    
    temporal_and_demand_analysis(df_sample, output_folder, max_heatmap_bins=5, dpi=80)
    capacity_and_utilization_analysis(df_sample, output_folder, max_heatmap_bins=5, dpi=80)
    cost_and_revenue_analysis(df_sample, output_folder, dpi=80)
    route_duration_coverage_analysis(df_sample, output_folder, max_heatmap_bins=5, dpi=80)
    
    print(f"\n✔ All route performance analysis visualizations saved under: {output_folder}\n")

# Temporal patterns and demand trends
def temporal_and_demand_analysis(df, out, max_heatmap_bins=8, dpi=80):
    """Analyze temporal patterns and demand trends, using only available KPIs."""
    # Column Chart: Riders per Hour
    plt.figure(figsize=(10,6))
    plt.bar(df["route_id"].astype(str), df["riders_per_hour"].fillna(0))
    plt.title("Riders per Hour by Route")
    plt.xlabel("Route ID")
    plt.ylabel("Riders per Hour")
    plt.xticks(rotation=90)
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(f"{out}/2.1_columnchart_riders_hour.png", dpi=dpi)
    plt.close()
    #Inference: Sorting routes by riders per hour highlights which routes are consistently busy versus underutilized.
    # This helps prioritize service improvements or additional resources for high-demand routes.
    
    # Peak vs Offpeak stacked area
    dfagg = df.copy()
    dfagg['peak_trips'] = dfagg['trips_per_day'] * (1 - dfagg['off_peak_trip_ratio'])
    dfagg['offpeak_trips'] = dfagg['trips_per_day'] * dfagg['off_peak_trip_ratio']

    dfagg = dfagg.sort_values('trips_per_day', ascending=False).head(min(40, len(dfagg)))

    plt.figure(figsize=(10,6))
    plt.stackplot(
        dfagg["route_id"].astype(str),
        dfagg["peak_trips"],
        dfagg["offpeak_trips"],
        labels=['Peak Trips', 'Off-Peak Trips']
    )

    plt.xlabel("Route ID")
    plt.ylabel("Trips per day") 
    plt.title("Peak vs Off-Peak Trip Distribution by Route")
    plt.xticks(rotation=90)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{out}/2.1_area_peak_offpeak.png", dpi=dpi)
    plt.close()
    #Inference: The stacked area shows how trips are distributed across peak and off-peak periods. Some routes carry most passengers during peak hours,
    #suggesting a focus on peak capacity, while others have steady demand throughout the day.


    # Heatmap: riders_per_hour by service_span vs route_duration
    try:
        heat = pd.pivot_table(
            df,
            values="riders_per_hour",
            index=pd.qcut(df["service_span"], q=min(max_heatmap_bins, len(df)), duplicates="drop"),
            columns=pd.qcut(df["route_duration"], q=min(max_heatmap_bins, len(df)), duplicates="drop"),
            aggfunc="mean"
        )
        plt.figure(figsize=(10,6))
        sns.heatmap(heat, cmap="YlOrRd")
        plt.title("Heatmap: Riders/Hour by Service Span vs Route Duration")
        plt.xlabel("Route Duration (hours)")
        plt.ylabel("Service Span (hours)")
        plt.tight_layout()
        plt.savefig(f"{out}/2.1_heatmap_weekly_pattern.png", dpi=dpi)
        plt.close()
    except Exception as e:
        print("Heatmap failed:", e)
    #Inference: This heatmap shows ridership patterns relative to route length and operating hours.
    # Medium-duration routes with moderate service spans tend to attract more riders, while very short or very long routes show lower utilization.
    

    # Strip/Box plot showing how riders per hour vary across service span durations.

    try:
        plt.figure(figsize=(12, 6))

        # Create service span bins (e.g., 8–10h, 10–12h, etc.)
        df["service_span_bin"] = pd.cut(
            df["service_span"],
            bins=6,
            labels=["Very Short", "Short", "Medium", "Long", "Very Long", "Extremely Long"]
        )

        # Strip plot
        sns.boxplot(
            data=df,
            x="service_span_bin",
            y="riders_per_hour",
            width=0.5,
            showfliers=False
        )

        sns.stripplot(
            data=df,
            x="service_span_bin",
            y="riders_per_hour",
            jitter=True,
            color="black",
            size=4,
            alpha=0.6
        )

        plt.title("Riders per Hour vs Service Span Category")
        plt.xlabel("Service Span Category (hours)")
        plt.ylabel("Riders per Hour")
        plt.tight_layout()
        plt.savefig(f"{out}/2.1_strip_boxplot_service_span_vs_ridership.png", dpi=dpi)
        plt.close()
    except Exception as e:
        print("2.1 Temporal plot failed:", e)

    #INFERENCE
    # Routes with longer service spans generally exhibit higher ridership per hour,
    # but variability increases significantly, indicating mismatches in off-peak supply.

    # Scatter: riders_per_trip vs operational_cost_per_rider
    plt.figure(figsize=(8,6))
    plt.scatter(df["riders_per_trip"], df["operational_cost_per_rider"], alpha=0.7)
    plt.xlabel("Riders per Trip")
    plt.ylabel("Operational Cost per Rider (CAD)")
    plt.title("Demand Vs Cost Relationship")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{out}/2.1_scatter_demand_cost.png", dpi=dpi)
    plt.close()
    #Inference: Routes with fewer riders per trip generally have higher cost per rider.
    # Efficient routes balance a high number of riders per trip with manageable costs.



In [10]:
# Capacity and utilization metrics
def capacity_and_utilization_analysis(df, out, max_heatmap_bins=8, dpi=80):
    #Load Factor Boxplot
    plt.figure(figsize=(6,6))
    sns.boxplot(y=df["load_factor_per_trip"])
    plt.title("Load Factor Distribution")
    plt.ylabel("Load Factor per Trip (%)")
    plt.tight_layout()
    plt.savefig(f"{out}/2.2_boxplot_loadfactor.png", dpi=dpi)
    plt.close()
    #Inference: Most routes operate below full capacity, indicating underutilized buses or scheduling mismatches.
    #Outliers show routes that are over-crowded or highly efficient.
    
    # Heatmap: Load Factor by Trips/Day vs Route Duration
    try:
        heat = pd.pivot_table(
            df,
            values="load_factor_per_trip",
            index=pd.qcut(df["trips_per_day"], q=min(max_heatmap_bins, len(df)), duplicates='drop'),
            columns=pd.qcut(df["route_duration"], q=min(max_heatmap_bins, len(df)), duplicates='drop'),
            aggfunc="mean"
        )
        plt.figure(figsize=(10,6))
        sns.heatmap(heat, cmap="Blues")
        plt.title("Utilization Heatmap")
        plt.xlabel("Route Duration (hours)")
        plt.ylabel("Trips per Day")
        plt.tight_layout()
        plt.savefig(f"{out}/2.2_heatmap_utilization.png", dpi=dpi)
        plt.close()
    except Exception as e:
        print("Utilization heatmap failed:", e)
    #Inference: Routes with frequent trips and medium durations show better utilization,
    # while very short or extremely long routes have lower efficiency. This helps in prioritizing scheduling adjustments.

    # Bubble chart: Trips/Day vs Riders/Trip
    plt.figure(figsize=(10,6))
    plt.scatter(
        df["trips_per_day"],
        df["riders_per_trip"],
        s=df["route_duration"] * 0.8,
        alpha=0.7
    )
    plt.xlabel("Trips per Day")
    plt.ylabel("Riders per Trip")
    plt.title("Utilization Efficiency (Bubble = Route Duration)")
    plt.tight_layout()
    plt.savefig(f"{out}/2.2_bubble_utilization.png", dpi=dpi)
    plt.close()
    #Inference: Longer routes tend to have fewer trips but higher riders per trip, whereas shorter routes may run more trips but carry fewer passengers per trip.
    # Bubble size indicates duration, making efficiency patterns easier to identify.



In [11]:
# Cost and revenue performance metrics
def cost_and_revenue_analysis(df, out, dpi=80):
    # Bar chart: Cost per Rider
    df_sorted = df.sort_values("operational_cost_per_rider")
    plt.figure(figsize=(12,6))
    plt.bar(df_sorted["route_id"].astype(str), df_sorted["operational_cost_per_rider"])
    plt.ylabel("Operational Cost per Rider (CAD)")
    plt.xlabel("Route ID")
    plt.title("Operational Cost per Rider")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f"{out}/2.3_bar_cost_per_rider.png", dpi=dpi)
    plt.close()
    #Inference: Shows which routes are the most expensive per passenger.
    #High-cost routes may need route restructuring or operational review to improve cost efficiency.


    # Treemap: Cost Contribution per Route
    try:
        treedf = df[["route_id","operational_cost_per_day"]].copy()
        treedf = treedf.groupby("route_id").sum().reset_index()

        sizes = treedf["operational_cost_per_day"]
        labels = [f"{r}\n{int(c):,}" for r, c in zip(treedf["route_id"], sizes)]

        plt.figure(figsize=(12,8))
        squarify.plot(sizes=sizes, label=labels, alpha=0.8)
        plt.title("Operational Cost Contribution (CAD)")
        plt.axis('off')
        plt.savefig(f"{out}/2.3_treemap_cost.png", dpi=dpi)
        plt.close()
    except:
        pass

    #Inference: A few routes account for most of the operational cost,
    #while the rest contribute minimally. This visualization helps focus cost-reduction efforts on major contributors.
    
    # Cost Efficiency Quadrant: Cost per Day vs Riders per Route
    plt.figure(figsize=(10,6))
    plt.scatter(
    df["operational_cost_per_day"].fillna(0),
    df["riders_per_route"].fillna(0),
    alpha=0.7
    )

    # Median reference lines to create quadrants
    plt.axvline(df["operational_cost_per_day"].median(), color="red", linestyle="--")
    plt.axhline(df["riders_per_route"].median(), color="red", linestyle="--")

    plt.xlabel("Operational Cost per Day (CAD)")
    plt.ylabel("Riders per Route")
    plt.title("Cost Efficiency Quadrant: Ridership vs Operational Cost")

    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{out}/2.3_quadrant_cost_efficiency.png", dpi=dpi)
    plt.close()

    # Inference:
    # Routes in the high-cost / low-ridership quadrant are underperforming and should be investigated.
    # Low-cost / high-ridership routes represent the most efficient services in the network.
 
    # Scatter: Riders/Trip vs Operational Cost/Day
    plt.figure(figsize=(10,6))
    plt.scatter(df["riders_per_trip"], df["operational_cost_per_day"])
    plt.xlabel("Riders per Trip")
    plt.ylabel("Operational Cost per Day (CAD)")
    plt.title("Cost-Productivity Relationship")
    plt.tight_layout()
    plt.savefig(f"{out}/2.3_scatter_cost_vs_riders.png", dpi=dpi)
    plt.close()
    #Inference: Routes carrying many riders per trip can still have high daily costs due to length or frequency,
    #highlighting that efficiency is influenced by multiple factors, not ridership alone.


In [12]:
# Route duration and service coverage patterns
def route_duration_coverage_analysis(df, out, max_heatmap_bins=8, dpi=80):

    # Scatter: Route Duration vs Riders/Route 
    plt.figure(figsize=(10,6))
    plt.scatter(df["route_duration"], df["riders_per_route"], marker="o")
    plt.xlabel("Route Duration (hours)")
    plt.ylabel("Riders per Route")
    plt.title("Route Duration vs Ridership")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{out}/2.4_scatter_duration_vs_riders.png", dpi=dpi)
    plt.close()
    #Inference: while some long routes attract fewer passengers, guiding decisions about route prioritization.
    
    # Bubble Chart: Route Duration vs Operational Cost (bubble = riders_per_hour, color = service_span)
    try:
        plt.figure(figsize=(10, 6))
        scatter = plt.scatter(
        df["route_duration"],
        df["operational_cost_per_day"],
        s=df["riders_per_hour"] * 5,   # bubble size scaled
        c=df["service_span"],          # color = service span
        alpha=0.6
        )

        plt.colorbar(scatter, label="Service Span (hours)")
        plt.xlabel("Route Duration (hours)")
        plt.ylabel("Operational Cost per Day (CAD)")
        plt.title("Route Duration vs Operational Cost with Ridership & Service Span")
        plt.tight_layout()
        plt.savefig(f"{out}/2.4_bubble_duration_cost.png", dpi=dpi)
        plt.close()

    except Exception as e:
        print("Bubble chart generation failed:", e)
    # Inference:
    # Longer route durations generally correlate with higher operational cost,but bubble size reveals whether this added cost is justified by ridership. Color gradient (service_span) exposes routes that run long hours but
    # underperform (large cost, small bubbles, long service_span).This visual highlights "inefficient" routes those with long duration,
    # long service span, high cost, but low riders per hour.

    # Boxplot: Service Span
    plt.figure(figsize=(8,6))
    sns.boxplot(y=df["service_span"])
    plt.ylabel("Service Span (hours)")
    plt.title("Service Span Distribution Across Routes")
    plt.tight_layout()
    plt.savefig(f"{out}/2.4_boxplot_service_span.png", dpi=dpi)
    plt.close()
    #Inference: Shows variation in operating hours across routes. 
    # Identifies routes with limited coverage that might require extended service, and those with long service spans that may have lower efficiency per hour.


In [13]:
#Standardization

def preprocess(df):
    """Standardize and prepare data for PCA. Fill missing KPI values with median."""
    df_clean = df.copy()
    #ensure all KPIs exist; if not, raise an informative error
    for col in KPI_LIST:
        if col not in df_clean.columns:
            raise RuntimeError(f"Required KPI not found in loaded data: {col}")
        df_clean[col] = df_clean[col].fillna(df_clean[col].median(skipna=True))
    #keep route_id as string and build scaled matrix for PCA
    df_clean['route_id'] = df_clean['route_id'].astype(str)
    route_ids = df_clean['route_id'].astype(str).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_clean[KPI_LIST].values)
    X_scaled_df = pd.DataFrame(X_scaled, index=route_ids, columns=KPI_LIST)
    return df_clean, scaler, X_scaled_df


In [14]:
# Principal Component Analysis (PCA)

def step4_pca(X_scaled_df, variance_threshold=0.88, min_components=3, max_components=5, plot=False):
    """Perform PCA and return PCA object, transformed PCs, loadings, and explained variance."""
    X = X_scaled_df.values
    n_features = X.shape[1]
    n_try = min(max_components, n_features)
    pca_full = PCA(n_components=n_try, random_state=42).fit(X)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    n_components = int(np.searchsorted(cumvar, variance_threshold) + 1)
    n_components = max(n_components, min_components)
    n_components = min(n_components, n_try)
    pca = PCA(n_components=n_components, random_state=42)
    pcs = pca.fit_transform(X)
    pcs_df = pd.DataFrame(pcs, index=X_scaled_df.index, columns=[f"PC{i+1}" for i in range(n_components)])
    loadings = pd.DataFrame(pca.components_.T, index=X_scaled_df.columns, columns=pcs_df.columns)
    explained = pd.DataFrame({
        "PC": pcs_df.columns,
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_variance": np.cumsum(pca.explained_variance_ratio_)
    })
    #save transformed PC values per route for auditing / reporting
    os.makedirs("results", exist_ok=True)
    pcs_df_reset = pcs_df.reset_index().rename(columns={'index':'route_id'})
    pcs_df_reset.to_csv("results/pca_routes.csv", index=False)
    pcs_df.to_csv("results/pca_transformed.csv")
    #return PCA artifacts
    return {
        "pca_obj": pca,
        "pcs_df": pcs_df,
        "loadings": loadings,
        "explained": explained,
        "n_components": n_components
    }


In [15]:
# K-Means Clustering & Interpretation
#----------------------------------------------------------
def step5_kmeans(pcs_df, k_min=4, k_max=6, random_state=42, plot=False, df_kpi=None):
    """
    Perform K-Means across k_min..k_max, select best K,
    and optionally generate scatter plot of internal clusters.
    Marker size = riders_per_route.
    """

    n_samples = pcs_df.shape[0]
    k_min = max(2, min(k_min, n_samples))
    k_max = max(k_min, min(k_max, n_samples))

    X = pcs_df.values
    results = []

    # Evaluate internal cluster counts
    for k in range(k_min, k_max + 1):
        km = KMeans(
            n_clusters=k, init='k-means++',
            n_init=20, max_iter=500,
            random_state=random_state
        )
        labels = km.fit_predict(X)
        inertia = km.inertia_
        sil = silhouette_score(X, labels) if len(np.unique(labels)) > 1 else -1
        results.append({'k': k, 'inertia': inertia, 'silhouette': sil})

    results_df = pd.DataFrame(results)

    # Best K selection
    if results_df['silhouette'].notna().any():
        best_idx = results_df['silhouette'].idxmax()
        best_k = int(results_df.loc[best_idx, 'k'])
        if best_k < 3 and any(results_df['k'] >= 3):
            cand_df = results_df[results_df['k'] >= 3]
            best_k = int(cand_df.loc[cand_df['silhouette'].idxmax(), 'k'])
    else:
        best_k = int(results_df.loc[results_df['inertia'].idxmin(), 'k'])

    # Final model
    final_model = KMeans(
        n_clusters=best_k, init='k-means++',
        n_init=20, max_iter=500,
        random_state=random_state
    )
    final_labels = final_model.fit_predict(X)

    # Append labels
    pcs_df_labeled = pcs_df.copy()
    pcs_df_labeled['cluster'] = final_labels

    #----------------------------------------------------------
    # INTERNAL CLUSTER SCATTER PLOT
    #----------------------------------------------------------
    if plot and df_kpi is not None:

        df_plot = pcs_df_labeled.copy()
        df_plot["route_id"] = df_plot.index.astype(str)

        df_plot = df_plot.merge(
            df_kpi[['route_id', 'riders_per_route']],
            on="route_id", how="left"
        )

        plt.figure(figsize=(10, 8))

        # Use a discrete palette sized to best_k (avoid mismatch between plotted colors and legend)
        palette = plt.cm.get_cmap('tab10', best_k)
        unique_labels = sorted(df_plot['cluster'].unique())
        color_map = {lab: palette(i) for i, lab in enumerate(unique_labels)}
        colors = df_plot['cluster'].map(color_map)

        scatter = plt.scatter(
            df_plot["PC1"], df_plot["PC2"],
            c=list(colors),
            s=np.sqrt(df_plot["riders_per_route"].fillna(1)) * 6,
            alpha=0.8
        )

        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.title(f"Internal K-Means Clusters (K={best_k}) — marker size ∝ riders_per_route")

        # Build a legend mapping cluster id -> color and count
        for lab in unique_labels:
            plt.scatter([], [], c=[color_map[lab]], label=f"Cluster {lab}")
        plt.legend(title="Cluster ID", bbox_to_anchor=(1.05, 1), loc='upper left')

        plt.grid(True)
        plt.tight_layout()
        plt.savefig("results/km_internal_clusters_scatter.png", dpi=150)
        plt.close()
    return {
        "best_k": best_k,
        "best_model": final_model,
        "pcs_labeled": pcs_df_labeled,
        "k_selection_df": results_df
    }


In [16]:
# All 12 KPIs are standardized and reduced using PCA, allowing the model to capture each route’s
# consolidated ridership, cost, and duration characteristics in fewer dimensions. K-Means is then
# applied on these PCA outputs to uncover internal clusters—groups of routes with similar overall
# performance profiles.
# For each cluster, we identify which principal components most strongly represent ridership
# (positive influence), cost (negative influence), and duration (negative influence). These
# directional impacts form the basis for computing composite scores and assigning final performance
# labels (High, Stable, Underperforming).These clusters represent structural similarities
# before applying final performance labels.

In [17]:
# INTERPRET & MAP INTO HIGH / STABLE / UNDERPERFORMING

def interpret_and_map_clusters(pca_obj, kmeans_model, pcs_labeled, df_kpi, kpi_cols, plot=False):
    """
    Compute composite score per internal cluster and map them
    to High / Stable / Underperforming tiers.
    """

    df_kpi = df_kpi.copy()
    df_kpi["route_id"] = df_kpi["route_id"].astype(str)

    pcs_labeled = pcs_labeled.copy()
    pcs_labeled.index = pcs_labeled.index.astype(str)

    # Loadings
    components = pca_obj.components_
    loadings_df = pd.DataFrame(
        components.T,
        index=kpi_cols,
        columns=[f"PC{i+1}" for i in range(components.shape[0])]
    )

    # Identify PC direction for “ridership / cost / duration”
    riders_kpi = ["riders_per_route", "riders_per_hour", "riders_per_trip"]
    cost_kpi = ["operational_cost_per_rider", "operational_cost_per_day"]
    duration_kpi = ["trip_duration", "route_duration"]

    def group_pc_and_sign(kpi_list):
        mean_abs = loadings_df.loc[kpi_list].abs().mean(axis=0)
        strongest_pc = mean_abs.idxmax()
        pc_idx = int(strongest_pc.replace("PC", "")) - 1
        sign = np.sign(loadings_df.loc[kpi_list][strongest_pc].mean())
        if sign == 0:
            sign = 1
        return pc_idx, float(sign)

    riders_pc, riders_sign = group_pc_and_sign(riders_kpi)
    cost_pc, cost_sign = group_pc_and_sign(cost_kpi)
    duration_pc, duration_sign = group_pc_and_sign(duration_kpi)

    centers_pc = kmeans_model.cluster_centers_


    # Composite scoring

    def minmax(x):
        x = np.asarray(x)
        return (x - x.min()) / (x.max() - x.min() + 1e-9)

    riders_vals = centers_pc[:, riders_pc] * riders_sign
    cost_vals = centers_pc[:, cost_pc] * cost_sign
    duration_vals = centers_pc[:, duration_pc] * duration_sign

    comp_vals = minmax(riders_vals) - minmax(cost_vals) - minmax(duration_vals)

    # Build cluster centers DF
    centers_df = pd.DataFrame(
        centers_pc,
        columns=[f"PC{i+1}" for i in range(centers_pc.shape[1])]
    )
    centers_df["cluster_id"] = list(range(len(centers_df)))
    centers_df["composite_score"] = comp_vals

    # Sort for ranking
    centers_df = centers_df.sort_values("composite_score", ascending=False).reset_index(drop=True)

    # Assign High / Stable / Underperforming
    k = len(centers_df)
    t1 = int(np.ceil(k / 3))
    t2 = int(np.ceil(2 * k / 3))

    centers_df["performance_label"] = "Stable"
    centers_df.loc[:t1-1, "performance_label"] = "High"
    centers_df.loc[t2:, "performance_label"] = "Underperforming"

    # Build mapping
    mapping = dict(zip(centers_df["cluster_id"], centers_df["performance_label"]))

    # Apply to pcs_labeled
    pcs_labeled["performance_label"] = pcs_labeled["cluster"].map(mapping)

    # Merge with KPI data
    merged = df_kpi.merge(
        pcs_labeled[["cluster", "performance_label"]].rename_axis("route_id").reset_index(),
        on="route_id", how="left"
    )


    # PLOT FINAL 3-PERFORMANCE LABELS
    
    if plot:

        df_plot = pcs_labeled.copy()
        df_plot["route_id"] = df_plot.index.astype(str)

        df_plot = df_plot.merge(
            df_kpi[["route_id", "riders_per_route"]],
            on="route_id", how="left"
        )

        color_map = {"High": "green", "Stable": "gold", "Underperforming": "red"}
        colors = df_plot["performance_label"].map(color_map)

        plt.figure(figsize=(9, 7))
        plt.scatter(
            df_plot["PC1"], df_plot["PC2"],
            c=colors,
            s=np.sqrt(df_plot["riders_per_route"].fillna(1)) * 6,
            alpha=0.8
        )
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.title("Performance Clusters Showing all TTC Routes\nMarker size ∝ riders_per_route")

        for lbl, col in color_map.items():
            plt.scatter([], [], c=col, label=lbl)
        plt.legend()

        plt.grid(True)
        plt.tight_layout()
        plt.savefig("results/final_clusters.png", dpi=150)
        plt.close()

    # Return artifacts
    return {
        "centers_df": centers_df,
        "loadings_df": loadings_df,
        "mapping": mapping,
        "pcs_labeled": pcs_labeled,
        "merged": merged,
        "riders_pc": riders_pc,
        "riders_sign": riders_sign,
        "cost_pc": cost_pc,
        "cost_sign": cost_sign,
        "duration_pc": duration_pc,
        "duration_sign": duration_sign
    }


In [18]:
#Composite Score And Clustering Interpretation Summary:
#A composite score is calculated for every cluster using:
# Higher ridership = higher score
# Higher cost = lower score
# Longer duration = lower score
# All cluster scores are normalized so they are comparable across the network.
#Clusters are then sorted by their composite score, and performance labels are assigned as follows:
#High performers (efficient, high ridership, low cost) → clusters in the top third of composite scores
#Stable performers(moderate performance) → clusters in the middle third
#Underperformers (low ridership, high cost, long duration)→ clusters in the bottom third


In [ ]:
#orchestrate the entire pipeline
#Fetching route names for descriptive exports
def fetch_route_names(route_ids):
    if len(route_ids) == 0:
        return pd.DataFrame(columns=['route_id', 'route_name', 'route_long_name'])

    # Normalize route IDs (remove spaces, ensure string)
    route_ids = pd.Series(route_ids).astype(str).str.strip().unique().tolist()
    sql = """
        SELECT 
            route_id::text AS route_id,
            route_name,
            route_long_name
        FROM gold."gold.dim_route"
        WHERE route_id::text = ANY(%s);
    """

    try:
        df = pd.read_sql_query(sql, engine, params=(route_ids,))
        df['route_id'] = df['route_id'].astype(str).str.strip()
        return df

    except Exception as e:
        print("⚠ Could not fetch route names from gold.dim_route:", e)
        return pd.DataFrame(columns=['route_id','route_name','route_long_name'])

def run_full_pipeline(
    variance_threshold=0.88,
    outlier_winsorize=True,
    outlier_lower_pct=0.01,
    outlier_upper_pct=0.99,
    internal_k_min=4,
    internal_k_max=6,
    run_route_analysis=True
):
    print("=" * 80)
    print("ROUTE ANALYSIS & CLUSTERING")
    print("=" * 80)

    #----------------------------------------------------------
    # STEP 1 — Load KPI data
    #----------------------------------------------------------
    print("\n[Step 1] Loading KPI data...")
    df = load_kpis()
    print(f"Data shape (raw): {df.shape}")
    print(f"Sample rows:\n{df.head()}\n")

    df_original = df.copy()

    #----------------------------------------------------------
    # STEP 2 — Outlier detection & winsorization
    #----------------------------------------------------------
    if outlier_winsorize:
        os.makedirs('results', exist_ok=True)

        print("[Step 2a] Outlier Detection (IQR Method)...")
        outlier_df = perform_outlier_detection(df)
        outlier_df.to_csv('results/outlier_detection_summary.csv', index=False)

        print("[Step 2b] Handling outliers with winsorization...")
        df_cleaned, winsor_df = handle_outliers_and_winsorize(
            df,
            lower_pct=outlier_lower_pct,
            upper_pct=outlier_upper_pct
        )
        winsor_df.to_csv('results/outlier_winsorization_summary.csv', index=False)
        print(f"✓ Winsorization applied. Total capped values = {int(winsor_df['Capped_Values'].sum())}")

        print("\n[Step 2c] Sensitivity Analysis...")
        sensitivity_df = perform_sensitivity_analysis(df_original, df_cleaned)
        sensitivity_df.to_csv('results/sensitivity_analysis.csv', index=False)

        df = df_cleaned

    #----------------------------------------------------------
    # STEP 3 — Preprocessing
    #----------------------------------------------------------
    print("\n[Step 3] Preprocessing & standardization...")
    df_kpi, scaler, X_scaled_df = preprocess(df)
    df_kpi["route_id"] = df_kpi["route_id"].astype(str)
    print(f"Standardized shape: {X_scaled_df.shape}")

    #----------------------------------------------------------
    # STEP 4 — Visuals
    #----------------------------------------------------------
    print("\n[Step 4] Generating route performance visualizations...")
    run_route_performance_analysis(df_kpi, output_folder="results/route_performance_visuals")

    #----------------------------------------------------------
    # STEP 5 — PCA
    #----------------------------------------------------------
    print("\n[Step 5] PCA transformation...")
    pca_res = step4_pca(X_scaled_df, variance_threshold=variance_threshold, min_components=3, max_components=5, plot=False)
    pcs_df = pca_res['pcs_df']; pca_obj = pca_res['pca_obj']
    pca_res['explained'].to_csv("results/pca_explained.csv", index=False)
    pca_res['loadings'].to_csv("results/pca_loadings.csv", index=True)
    print(f"PCA: selected n_components = {pca_res['n_components']}")
    print(f"Cumulative variance explained: {pca_res['explained']['cumulative_variance'].iloc[-1]:.2%}")
    # Print which PCs were selected and top contributing KPIs per PC
    try:
        n_comp = int(pca_res['n_components'])
        selected_pcs = [f"PC{i+1}" for i in range(n_comp)]
        print("Selected principal components:", ", ".join(selected_pcs))
        loadings = pca_res.get('loadings')
        if loadings is not None:
            print('\nTop contributing KPIs per selected PC :')
            for pc in selected_pcs:
                if pc in loadings.columns:
                    top_feats = loadings[pc].abs().sort_values(ascending=False).head(3)
                    formatted = [f"{feat} ({loadings[pc].loc[feat]:+.3f})" for feat in top_feats.index]
                    print(f"  {pc}: " + ", ".join(formatted))
    except Exception as e:
        print("Could not print selected PCs/loadings:", e)

    #----------------------------------------------------------
    # STEP 6 — Internal KMeans Clustering
    #----------------------------------------------------------
    print("\n[Step 6] K-Means clustering (internal clusters)...")
    k_res = step5_kmeans(
        pcs_df,
        k_min=internal_k_min,
        k_max=internal_k_max,
        plot=True,       # Produce internal scatter plot
        df_kpi=df_kpi
    )

    best_k = k_res['best_k']
    best_model = k_res['best_model']
    pcs_labeled = k_res['pcs_labeled']
    k_selection = k_res['k_selection_df']
    k_selection.to_csv("results/k_selection.csv", index=False)

    print(f"✓ Internal K selected = {best_k}")

    #----------------------------------------------------------
    # STEP 7 — Interpret clusters (High / Stable / Underperforming)
    #----------------------------------------------------------
    print("\n[Step 7] Cluster interpretation & mapping...")
    interp = interpret_and_map_clusters(
        pca_obj,
        best_model,
        pcs_labeled,
        df_kpi,
        KPI_LIST,
        plot=True   # Produce final performance scatter plot
    )

    centers_df = interp["centers_df"]
    mapping = interp["mapping"]
    pcs_labeled = interp["pcs_labeled"]
    merged = interp["merged"]

    centers_df.to_csv("results/cluster_centers_internal.csv", index=False)

    print("\nCluster → Performance mapping:")
    for cid, lbl in mapping.items():
        print(f"  Internal cluster {cid} → {lbl}")

    #----------------------------------------------------------
    # STEP 8 — Route-level composite scoring
    #----------------------------------------------------------
    print("\n[Step 8] Computing per-route composite scores...")

    rpc = interp['riders_pc'];  rsg = interp['riders_sign']
    cpc = interp['cost_pc'];    csg = interp['cost_sign']
    dpc = interp['duration_pc']; dsg = interp['duration_sign']

    def minmax(x):
        x = np.asarray(x)
        return (x - x.min()) / (x.max() - x.min() + 1e-9)

    # Compute raw component contributions
    riders_vals = pcs_df[f"PC{rpc+1}"] * rsg
    cost_vals = pcs_df[f"PC{cpc+1}"] * csg
    duration_vals = pcs_df[f"PC{dpc+1}"] * dsg

    # Compute composite → convert to Series so naming works
    composite_array = minmax(riders_vals) - minmax(cost_vals) - minmax(duration_vals)
    composite = pd.Series(composite_array, index=pcs_df.index, name="composite_score")

    merged = merged.set_index("route_id")
    merged.loc[composite.index, "composite_score"] = composite
    merged = merged.reset_index()

   
    # STEP 9 — Export results
    #----------------------------------------------------------
    print("\n[Step 9] Exporting results...")

    # 1. Fetch route name mapping
    route_names_df = fetch_route_names(
    merged['route_id'].astype(str).str.strip().unique().tolist()
    )

    # 2. Merge into "merged"
    if not route_names_df.empty:
        merged['route_id'] = merged['route_id'].astype(str).str.strip()
        merged = merged.merge(route_names_df, on='route_id', how='left')
    else:
        merged['route_name'] = np.nan
        merged['route_long_name'] = np.nan

    # 3. Reorder columns
    cols = merged.columns.tolist()
    front = ['route_id', 'route_name', 'route_long_name']
    rest = [c for c in cols if c not in front]
    merged = merged[front + rest]

    # 4. Save outputs
    os.makedirs("results", exist_ok=True)
    
    # Save full results with error handling
    try:
        merged.to_csv("results/ttc_routes_with_clusters_and_scores.csv", index=False)
        print(f"✓ Saved full results ({len(merged)} routes)")
    except PermissionError:
        print("⚠ WARNING: Could not save 'ttc_routes_with_clusters_and_scores.csv'")
        print("  The file may be open in Excel or another program. Please close it and try again.")
        print("  Continuing with analysis...")

    # --- CREATE under_df HERE so it exists for later use ---
    under_df = merged[merged["performance_label"] == "Underperforming"].copy()
    
    # Save underperforming routes with error handling
    try:
        under_df.to_csv("results/underperforming_routes.csv", index=False)
        print(f"✓ Saved underperforming ({len(under_df)} routes)")
    except PermissionError:
        print("⚠ WARNING: Could not save 'underperforming_routes.csv'")
        print("  The file may be open in Excel or another program. Please close it and try again.")
        print("  Continuing with analysis...")

    #----------------------------------------------------------
    # DISPLAY TOP UNDERPERFORMING ROUTES
    #----------------------------------------------------------
    if not under_df.empty:
        print("\n Sample of underperforming routes:")
        
        # Build display columns based on what's available
        display_cols = ["route_id"]
        if 'route_name' in under_df.columns:
            display_cols.append("route_name")
        if 'route_long_name' in under_df.columns:
            display_cols.append("route_long_name")
        display_cols.extend(["cluster", "performance_label", "composite_score"])
        
        print(
            under_df.sort_values("composite_score")
                    .head(10)[display_cols].to_string(index=False)
        )

# Save cluster centers
    try:
        centers_df.to_csv("results/cluster_centers_summary.csv", index=False)
    except PermissionError:
        print("⚠ WARNING: Could not save 'cluster_centers_summary.csv'")
        print("  The file may be open in Excel or another program.")

    return {
        "merged": merged,
        "underperforming": under_df,
        "pcs_labeled": pcs_labeled,
        "centers_df": centers_df,
        "pca_res": pca_res,
        "k_selection": k_selection
        }

#Main block execution
if __name__ == "__main__":
    results = run_full_pipeline(
        variance_threshold=0.88,
        outlier_winsorize=True,
        outlier_lower_pct=0.01,
        outlier_upper_pct=0.99,
        internal_k_min=4,
        internal_k_max=6,
        run_route_analysis=True
    )
    print("\n" + "=" * 80)
    print("Underperforming Routes identified successfully.")
    print("=" * 80)


ROUTE ANALYSIS & CLUSTERING

[Step 1] Loading KPI data...
Data shape (raw): (189, 13)
Sample rows:
  route_id  riders_per_route  riders_per_hour  riders_per_trip  \
0    74750             520.0        52.000000        12.380952   
1    74751           10018.0        47.933014        32.845902   
2    74752             815.0        47.941176        10.187500   
3    74753           23548.0        50.859611        56.200477   
4    74754            2215.0        58.289474        23.817204   

   operational_cost_per_rider  operational_cost_per_day  trip_duration  \
0                    5.288462                    2750.0       0.227116   
1                    5.737173                   57475.0       0.560722   
2                    5.736196                    4675.0       0.172260   
3                    5.407041                  127325.0       1.010610   
4                    4.717833                   10450.0       0.430699   

   route_duration  load_factor_per_trip  trips_per_day  hou